In [46]:
import sys
print(sys.executable)
import src
print("ok")
#!pip install ftfy

c:\Users\DiDi\miniconda3\envs\py311_new\python.exe
ok


In [47]:
import pandas as pd
import re

def find_irregular_characters(df: pd.DataFrame, column: str):
    """
    Busca caracteres sospechosos de problemas de encoding,
    ignorando puntuación normal como ?, ., ,, :, ;, !, etc.
    """

    def analyze_row(text):
        if not isinstance(text, str):
            return None
        
        # Caracteres típicos de encoding roto / mojibake
        irregular_pattern = r'[â€™œžŸšŠ¢�ÃÂ]|[\u0080-\u009F]'
        
        results = []
        words = text.split()
        seen_chars = set()
        
        for word in words:
            matches = re.findall(irregular_pattern, word)
            
            for char in matches:
                if char not in seen_chars:
                    results.append(f"'{char}' en '{word}'")
                    seen_chars.add(char)
        
        return ", ".join(results) if results else "Ninguno"

    new_col_name = f"irregulares_{column}"
    df[new_col_name] = df[column].apply(analyze_row)
    
    return df


In [48]:
import src.data_ingestion as data_loader

data = "../data/data_test_fold1.csv"

columnas = ["user_id","text_id","title","text","is_suicide"]

df = data_loader.data_loader(data, columnas)
df = data_loader.data_mapping(df, "is_suicide", "yes", "no")
df.isna().sum()
#ver filas vacias
#df[df.isna().any(axis=1)]

Data Loaded, schema valid


user_id       0
text_id       0
title         0
text          1
is_suicide    0
dtype: int64

Decidí solo concateran los titulos con los textos

In [49]:
# concatenate
df = data_loader.concatenate_df(df, ["title","text"])
#df

In [50]:
df = find_irregular_characters(df, 'title_text')
print(df[['irregulares_title_text']])

                                irregulares_title_text
0                                              Ninguno
1                                              Ninguno
2                                              Ninguno
3    'â' en 'Nothingâ€™s', '€' en 'Nothingâ€™s', '™...
4                                              Ninguno
..                                                 ...
247                                            Ninguno
248                                            Ninguno
249  'â' en 'canâ€™t', '€' en 'canâ€™t', '™' en 'ca...
250                                            Ninguno
251  'â' en 'itâ€™s', '€' en 'itâ€™s', '™' en 'itâ€...

[252 rows x 1 columns]


In [51]:
from ftfy import fix_text

df['title_text_clean'] = df['title_text'].apply(fix_text)
df['title_text_clean']

0      I crave the eternal silence and slumber that c...
1      I dont know what to write here To kill myself ...
2      Every dream I've ever had has been crushed. I'...
3      Nothing's changed, hope is gone, I guess I got...
4      Am i the only one who doesn't understand this?...
                             ...                        
247    I just really wish I had died the first time I...
248    I feel unimportant. Not the first time I'm goi...
249    What should I do I feel like people are contro...
250                      How do I kill myself? [removed]
251    My dad is offering us ice cream now Every nigh...
Name: title_text_clean, Length: 252, dtype: str